### Will, precisamos iniciar a modelagem dos dados dos nossos clientes, mas a equipe de visualização não pode acessar o arquivo cru no Data Lake.
### Preciso que você crie um banco de dados focado em CRM e ingira aquele arquivo CSV de Clientes como a nossa tabela bruta oficial (raw) para que possamos plugar o Power BI.


As Regras de Negócio e Metadados da Ingestão:

O Arquivo Exato: O caminho do seu arquivo é dbfs:/Volumes/workspace/default/arq-aula-dbs/arquivos_csv/Clientes.csv.

O Delimitador: Baseado no arquivo que você carregou, preste muita atenção ao delimitador. Se os dados estiverem separados por vírgula no arquivo bruto, a opção deverá refletir isso (e cuidado com o erro de digitação da palavra delimiter que mapeamos na auditoria anterior).

A Estrutura de Destino (SQL): Crie um schema novo e isolado chamado CRM_ANALYTICS.

A Persistência (PySpark): Leia o arquivo CSV ativando o reconhecimento de cabeçalho (header) e a inferência de tipos (inferSchema). Salve o DataFrame como uma tabela chamada clientes_raw dentro do novo schema CRM_ANALYTICS.

Idempotência: Garanta que a sua escrita use o modo overwrite (sobrescrever), para que reexecutar o bloco não duplique a base inteira de clientes.

In [0]:
%fs
ls dbfs:/

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
%fs
ls '/Volumes/workspace/default/arq-aula-dbs/'

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/arq-aula-dbs/Anac/,Anac/,0,1778188016462
dbfs:/Volumes/workspace/default/arq-aula-dbs/Bike Store/,Bike Store/,0,1778188016462
dbfs:/Volumes/workspace/default/arq-aula-dbs/arquivos_csv/,arquivos_csv/,0,1778188016462


In [0]:
%sql
CREATE SCHEMA WORKSPACE.CRM_ANALYTICS

- criar um db focado em CRM (CRM_ANALYTICS).
Criar uma tabela chamada clientes_raw dentro do Db (CRM_ANALYTICS).

In [0]:
file_path = 'dbfs:/Volumes/workspace/default/arq-aula-dbs/arquivos_csv/Clientes.csv'
 

df_cliente_raw = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("delimiter", ",")\
    .load(file_path)
df_cliente_raw.write.mode('overwrite').saveAsTable('WORKSPACE.CRM_ANALYTICS.clientesRaw')


In [0]:
%sql
select * from workspace.crm_analytics.clientesraw 




id,created_at,first_name,last_name,email,cell_phone,country,state,street,number,additionals
0,2017-11-01T14:45:41.000Z,Marta,Jesus,null,9 9102-7834,Brasil,Acre,null,null,Conjunto 16
1,2017-10-16T00:50:39.000Z,Luana,Almeida,null,9 7328-8718,Brasil,Rio Grande do Sul,Avenida 56 do Estado Rio Grande do Sul,989.0,Conjunto 17
2,2018-06-16T17:51:29.000Z,Frida,Mendes,frida@meu_email.com,9 5906-7552,Brasil,São Paulo,Avenida 59 do Estado São Paulo,534.0,null
3,2018-01-17T03:02:58.000Z,Daniela,Avelino,daniela@exemplo.com,9 4642-9486,Brasil,Mato Grosso,null,null,null
4,2018-08-06T07:24:16.000Z,Romário,Teixeira,null,9 3093-6522,Brasil,Bahia,Praça 56 do Estado Bahia,191.0,Apto 12
5,2018-01-05T17:20:49.000Z,Marcelo,Barroso,null,9 2830-2088,Brasil,Rio Grande do Sul,Rua 28 do Estado Rio Grande do Sul,805.0,Conjunto 13
6,2018-06-18T11:17:42.000Z,Cristiano,Elísio,cristiano@exemplo.com,9 3532-8404,Brasil,Goiás,Rua 78 do Estado Goiás,877.0,Apto 14
7,2018-02-08T12:36:09.000Z,Everton,Barbosa,everton@meu_email.com,9 2553-4087,Brasil,Distrito Federal,Avenida 86 do Estado Distrito Federal,864.0,Apto 14
8,2017-12-16T20:47:03.000Z,Gabriela,Alves,gabriela@exemplo.com,9 1353-8433,Brasil,Santa Catarina,null,null,null
9,2018-11-11T11:48:41.000Z,Luan,Dias,luan@exemplo.com,9 2417-3678,Brasil,Distrito Federal,Avenida 54 do Estado Distrito Federal,889.0,Conjunto 14


**Will, excelente trabalho na ingestão da base bruta. Agora precisamos de uma versão limpa dessa tabela para plugar nas nossas ferramentas operacionais.
O nosso sistema de disparo de marketing quebra se receber um campo de email como 'null'. Além disso, o sistema de roteirização da logística não aceita 'null' na coluna de número da casa, ele precisa de um número real.**

**1.Alterar os valores de email de nulo para nao informado.

 2.Alterar os valores de casa(additionals) null para 0.

 3. Salvar df como uma nova tabela chamada cliente_silver, dentro do schema.workspace.CRM_ANALYTICS.**



In [0]:
%sql
CREATE TABLE WORKSPACE.crm_analytics.cliente_silver

In [0]:
df_clientes_silver = spark.read.table('workspace.crm_analytics.clientesraw')\
    .na.fill({'state':'estado_nao_informado', 'email':'nao_informado', 'street':'nao_informado', 'number': 0, 'additionals': 'sem_complemento' })
df_clientes_silver.write.mode('overwrite').saveAsTable('workspace.crm_analytics.cliente_silver')

In [0]:
%sql
select * from workspace.crm_analytics.cliente_silver

id,created_at,first_name,last_name,email,cell_phone,country,state,street,number,additionals
0,2017-11-01T14:45:41.000Z,Marta,Jesus,nao_informado,9 9102-7834,Brasil,Acre,nao_informado,0.0,Conjunto 16
1,2017-10-16T00:50:39.000Z,Luana,Almeida,nao_informado,9 7328-8718,Brasil,Rio Grande do Sul,Avenida 56 do Estado Rio Grande do Sul,989.0,Conjunto 17
2,2018-06-16T17:51:29.000Z,Frida,Mendes,frida@meu_email.com,9 5906-7552,Brasil,São Paulo,Avenida 59 do Estado São Paulo,534.0,sem_complemento
3,2018-01-17T03:02:58.000Z,Daniela,Avelino,daniela@exemplo.com,9 4642-9486,Brasil,Mato Grosso,nao_informado,0.0,sem_complemento
4,2018-08-06T07:24:16.000Z,Romário,Teixeira,nao_informado,9 3093-6522,Brasil,Bahia,Praça 56 do Estado Bahia,191.0,Apto 12
5,2018-01-05T17:20:49.000Z,Marcelo,Barroso,nao_informado,9 2830-2088,Brasil,Rio Grande do Sul,Rua 28 do Estado Rio Grande do Sul,805.0,Conjunto 13
6,2018-06-18T11:17:42.000Z,Cristiano,Elísio,cristiano@exemplo.com,9 3532-8404,Brasil,Goiás,Rua 78 do Estado Goiás,877.0,Apto 14
7,2018-02-08T12:36:09.000Z,Everton,Barbosa,everton@meu_email.com,9 2553-4087,Brasil,Distrito Federal,Avenida 86 do Estado Distrito Federal,864.0,Apto 14
8,2017-12-16T20:47:03.000Z,Gabriela,Alves,gabriela@exemplo.com,9 1353-8433,Brasil,Santa Catarina,nao_informado,0.0,sem_complemento
9,2018-11-11T11:48:41.000Z,Luan,Dias,luan@exemplo.com,9 2417-3678,Brasil,Distrito Federal,Avenida 54 do Estado Distrito Federal,889.0,Conjunto 14


** Will, a base de clientes limpa (Silver) ficou excelente. Na semana que vem, vamos lançar uma grande campanha de anúncios regionais e o orçamento é limitado.
Preciso que você crie uma tabela consolidada que me mostre quantos clientes nós temos em cada estado. E para facilitar a minha vida no dashboard, essa tabela já tem que vir ordenada, mostrando os estados com mais clientes no topo.**

1. Criar uma tabela que mostre quantidade de clientes por estado.
2. Agrupar os dados da coluna state
3. Ordenar pro do estado maior para o menor.
4. Criar a tabela cliente_gold, dentro do crm_anaytics.

In [0]:
%sql
CREATE TABLE WORKSPACE.crm_analytics.clientes_estado_gold

In [0]:
%sql
drop table WORKSPACE.crm_analytics.clientes_estado_gold

In [0]:
from pyspark.sql.functions import desc, col

df_clientes_gold = spark.read.table('workspace.crm_analytics.cliente_silver')\
    .groupBy("state").count()\
    .orderBy(col("count").desc())
df_clientes_gold.write.mode('overwrite').saveAsTable('workspace.crm_analytics.clientes_estado_gold')


In [0]:
%sql
select * from workspace.crm_analytics.clientes_estado_gold

state,count
Pernambuco,8
Rio Grande do Sul,6
Bahia,6
Maranhão,6
Ceará,5
Goiás,5
Mato Grosso,5
Rio Grande do Norte,5
Minas Gerais,4
São Paulo,4
